In [4]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from pydantic import BaseModel , Field
import os   
import operator

In [80]:
load_dotenv()

True

In [65]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(
        description="Detailed feedback only. Do NOT include the score here."
    )
    score: int = Field(
        description="A single integer score between 0 and 10. Must be separate from feedback.",
        ge=0,
        le=10
    )

In [81]:
llm = ChatOpenAI(model="openai/gpt-oss-120b")
structureLLM = llm.with_structured_output(EvaluationSchema)

In [67]:
class ExamState(TypedDict):
    essay:str
    language_feedback:str
    analysis_feedback:str
    clarity_feedback:str
    overall_feedback:str
    individual_score:Annotated[list[int],operator.add]
    avg_score:float

In [68]:
def eval_lang(state:ExamState):
    prompt=f"Evaluate the language quality of the following essay and provide a feedback and assign a score between 0 to 10 \n {state['essay']}"    
    output = structureLLM.invoke(prompt)
    return{"language_feedback":output.feedback,"individual_score":[output.score]}

In [69]:
def eval_analysis(state:ExamState):
    prompt=f"Evaluate the depth of the analysis of the following essay and provide a feedback and assign a score between 0 to 10 \n {state['essay']}"    
    output = structureLLM.invoke(prompt)
    return{"analysis_feedback":output.feedback,"individual_score":[output.score]}

In [70]:
def eval_clarity(state:ExamState):
    prompt=f"Evaluate the clarity of thought of the following essay and provide a feedback and assign a score between 0 to 10 \n {state['essay']}"    
    output = structureLLM.invoke(prompt)
    return{"clarity_feedback":output.feedback,"individual_score":[output.score]}

In [71]:
def final_eval(state:ExamState):
    #summary feedback
    prompt = f"based on the following feedback create a summarized feedback \n language feedback {state['language_feedback']} \n depth of analysis feedback :{state['analysis_feedback']} \n clarity of thought feedback :{state['clarity_feedback']}"
    overallFeedback=llm.invoke(prompt).content
    #avg calculation
    avg = sum(state['individual_score'])/len(state['individual_score'])
    return {"overall_feedback":overallFeedback,"avg_score":avg}

In [72]:
graph=StateGraph(ExamState)

graph.add_node("evaluate_language",eval_lang)
graph.add_node("evaluate_analysis",eval_analysis)
graph.add_node("evaluate_clarity",eval_clarity)
graph.add_node("final_evaluation",final_eval)

graph.add_edge(START,"evaluate_language")
graph.add_edge(START,"evaluate_analysis")
graph.add_edge(START,"evaluate_clarity")
graph.add_edge("evaluate_language","final_evaluation")
graph.add_edge("evaluate_analysis","final_evaluation")
graph.add_edge("evaluate_clarity","final_evaluation")
graph.add_edge("final_evaluation",END)


In [73]:
workflow = graph.compile()

In [ ]:
essay="""Artificial Intelligence (AI) is one of the most transformative technologies of the modern era. It refers to the ability of machines to perform tasks that typically require human intelligence, such as learning, reasoning, problem-solving, and decision-making. From simple automation to complex systems that can understand language and recognize images, AI is reshaping the way people live and work.

The concept of AI is not entirely new. Early ideas about intelligent machines date back to ancient times, but the field began to take shape in the 20th century with the development of computers. Today, AI has advanced rapidly due to increased computing power, large amounts of data, and improved algorithms. Technologies like machine learning and deep learning allow systems to learn from experience and improve over time without being explicitly programmed."""
intial_state ={"essay": essay}
output = workflow.invoke(intial_state)

print(f"languuage feedback - {output['language_feedback']}" )
print(f"Analysis feedback - {output['analysis_feedback']}")
print(f"Clarity feedback - {output['clarity_feedback']}")
print(f"Overall feedback - {output['overall_feedback']}")
print(f"individiual score -{output['individual_score']}")
print(f"Score- {output['avg_score']}")

languuage feedback - The essay provides a clear and concise introduction to the concept of Artificial Intelligence, its history, and its current advancements. The language is formal and suitable for an academic or informative piece. The structure is logical, starting with a broad definition and then narrowing down to specific aspects such as the role of computing power, data, and algorithms in AI's development. The use of transitional phrases and sentences helps to maintain a smooth flow of ideas. However, the essay could benefit from more nuanced and detailed explanations of key terms like machine learning and deep learning, as well as examples to illustrate the practical applications and implications of AI. Overall, the writing demonstrates a good command of the subject matter and effective communication skills.
Analysis feedback - The essay provides a good introduction to the concept of Artificial Intelligence, its history, and its current advancements. However, the analysis is quit